In [ ]:
from reedsolo import RSCodec, rs_calc_syndromes
import os
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from rs.channels import qsc_erasure_channel
from rs.dataset_gen import RSPositionDataset, bytes_to_bits, bits_to_bytes, get_zero_mask

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class PositionPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(511, 512),
            nn.ReLU(),

            nn.Linear(511, 512),
            nn.ReLU(),

            nn.Linear(511, 512),
            nn.ReLU(),

            nn.Linear(511, 512),
            nn.ReLU(),

            nn.Linear(512, 255)
        )
    
    def forward(self, x):
        return self.net(x)

In [ ]:
class HybridDecoder:
    def __init__(self, model, threshold=0.3):
        self.model = model
        self.threshold = threshold
        self.rsc = RSCodec(32)
    
    def decode(self, noisy):
        syndrome = rs_calc_syndromes(noisy, 32)[1:]
        syndrome_bits = bytes_to_bits(bytes(syndrome))
        zero_mask = get_zero_mask(noisy)
        inp = np.concatenate([syndrome_bits, zero_mask]).astype(np.float)

        x = torch.tensor(inp).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = self.model(x)
            probs = torch.sigmoid(logits)
        
        positions = (probs[0] > self.threshold).cpu().numpy()
        erase_pos = [i for i, v in enumerate(positions) if v]

        try:
            decoded, _, _ = self.rsc.decode(noisy, erase_pos=erase_pos)
            return bytes(decoded)
        except:
            return None

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def evaluate(model, p_err, p_erase, num_samples=1000):
    model.eval()
    rsc = RSCodec(32)
    hybrid = HybridDecoder(model)

    classic_no_hint = 0
    classic_with_hint = 0
    hybrid_success = 0

    for _ in range(num_samples):
        msg = os.urandom(223)
        codeword = rsc.encode(msg)
        noisy, erasure_pos = qsc_erasure_channel(codeword, p_err, p_erase)

        try:
            decoded, _, _ = rsc.decode(noisy)
            if bytes(decoded) == msg:
                classic_no_hint += 1
        except:
            pass

        try:
            decoded, _, _ = rsc.decode(noisy, erase_pos=erasure_pos)
            if bytes(decoded) == msg:
                classic_with_hint += 1
        except:
            pass

        decoded = hybrid.decode(noisy)
        if decoded == msg:
            hybrid_success += 1
    
    return {
        'classic': classic_no_hint / num_samples,
        'hybrid': hybrid_success / num_samples,
        'classic_hint': classic_with_hint / num_samples
    }